# **Exercise 1: Segmentation and Tokenization**

In this exercise, you will work with concepts from Chapter 2 of *Speech and Language Processing*:

1. regular expressions and word tokenization;
2. minimum edit distance; and
3. subword tokenization with Byte-Pair Encoding (BPE).

We're going to implement the algorithms and analyze their strengths and weaknesses.


In [ ]:
%pip install -q datasets tokenizers

import re
from collections import Counter

import datasets

datasets.disable_progress_bars()
datasets.logging.set_verbosity_error()


## **Ex. 1: Regular expressions and word tokenization**

Regular expressions describe patterns in strings. Python's [`re`](https://docs.python.org/3/library/re.html) module can use them to search, extract, split, and replace text.

Use raw strings such as `r"\bword\b"` for regular expressions. This prevents Python's string parser from interpreting backslashes before the regular-expression engine sees them.

Useful references:

1. [Python regular-expression documentation](https://docs.python.org/3/library/re.html)
2. [Regex101](https://regex101.com/)
3. Section 2.6 of *Speech and Language Processing*


In [ ]:
# Backslashes escape sequences in ordinary Python strings
print("A newline follows this colon:\nSecond line")
print("A literal backslash needs escaping: \\")

# A raw string does treats backslashes as literal characters,
# which is convenient for regex patterns.
print(r"A raw string contains the characters \n rather than a newline.")


The most common matching functions are:

- `re.match`: match only at the start of a string;
- `re.search`: find the first match anywhere in a string; and
- `re.findall`: return all non-overlapping matches.

A `re.Match` object stores information about a match. Its matched text is available through `.group()`.


In [ ]:
import re

text = """The Solar System is about 4.568 billion years old.[1]
The Sun formed by gravity in a large molecular cloud.
It is mainly hydrogen, which it converts into helium through nuclear fusion.
The planets are in a flattened orbiting disk. This disk was partly left over
from the cloud that formed the Sun, plus other material as the Sun moved through space.
Eventually, the gas and dust of the disk came together into planets.
It is thought that almost all stars and their planets form this way.

The Sun is about a hundred times as wide as the Earth.
It has a mass of 1.9891x1030 kg.
This is 333,000 times the mass of the Earth.
1.3 million Earths can fit inside the Sun.[16]
The Sun fuses about 600 million tons of hydrogen into helium every second.
"""

any_uppercase = r"[A-Z]"
not_any_uppercase = r"[^A-Z]"
any_alphanumeric = r"\w+"
ending_in_illion_or_gen = r".illion|\b\w+gen"


# Check if at least one occurrence is in the string
print("-- Search --")
print(re.search(any_uppercase, text).group())
print(re.search(not_any_uppercase, text).group())
print(re.search(any_alphanumeric, text).group())
print(re.search(ending_in_illion_or_gen, text).group())


# Find the first occurrence at the start of the string and stop
print("\n -- Match --")
print(re.match(any_uppercase, text).group())
print(re.match(not_any_uppercase, text))
print(re.match(any_alphanumeric, text).group())
print(re.match(ending_in_illion_or_gen, text))

# Greedily ask for all matches (only showing 10)
print("\n -- Findall --")
print(re.findall(any_uppercase, text)[:10])
print(re.findall(not_any_uppercase, text)[:10])
print(re.findall(any_alphanumeric, text)[:10])
print(re.findall(ending_in_illion_or_gen, text)[:10])

-- Search --
T
h
The
billion

 -- Match --
T
None
The
None

 -- Findall --
['T', 'S', 'S', 'T', 'S', 'I', 'T', 'T', 'S', 'S']
['h', 'e', ' ', 'o', 'l', 'a', 'r', ' ', 'y', 's']
['The', 'Solar', 'System', 'is', 'about', '4', '568', 'billion', 'years', 'old']
['billion', 'hydrogen', 'million', 'million', 'hydrogen']


### Regex practice

Write regular expressions for the following tasks:

1. Return all words ending in **s**.
2. Return all words immediately preceding a word starting with **s** or **h**.
3. Return the references `[1]` and `[16]`.
4. Count sentence-final punctuation marks. Do not count the decimal point in `4.568`.
5. Return numbers followed by a unit, such as `4.568 billion` or `600 million`.

There can be several correct solutions.

<div class="alert alert-block alert-info">
<b>TIME:</b> About 20 minutes.
</div>


In [ ]:
# Your solution here

In [ ]:
print(re.findall(r"\b\w+s\b", text, flags=re.IGNORECASE))
print(re.findall(r"\b(\w+)\s+(?=[sh]\w*\b)", text, flags=re.IGNORECASE))
print(re.findall(r"\[\d+\]", text))
print(len(re.findall(r"[.!?](?=\s|\[|$)", text)))
print(re.findall(r"\b\d[\d,.]*(?:x\d+)?\s+\w+", text))


### Splitting and Tokenization

Models require text to be represented as a sequence of units. We need to choose which units we're going to use.
A first approach is to split on whitespace. Inspect the output and identify examples of:

- **undersplitting**, where several useful units remain in one token; and
- **oversplitting**, where one useful unit is split into several tokens.

Think about the consequences for morphological analysis, vocabulary lookup, and counting.


In [ ]:
whitespace_tokens = re.split(r"\s+", text)
print(whitespace_tokens[:30])


['The', 'Solar', 'System', 'is', 'about', '4.568', 'billion', 'years', 'old.[1]', 'The', 'Sun', 'formed', 'by', 'gravity', 'in', 'a', 'large', 'molecular', 'cloud.', 'It', 'is', 'mainly', 'hydrogen,', 'which', 'it', 'converts', 'into', 'helium', 'through', 'nuclear']


Now construct the target vocabulary listed below in three steps:

1. normalize the text;
2. tokenize it; and
3. count the tokens.

For this exercise, numbers, punctuation, references, and one-character words are excluded. Real tokenization decisions depend on the application; this target is only one possible convention.

<div class="alert alert-block alert-info">
<b>TIME:</b> About 15 minutes.
</div>


In [ ]:
from collections import Counter

target_vocabulary = {
    "about": 3,
    "all": 1,
    "almost": 1,
    "and": 2,
    "are": 1,
    "as": 3,
    "billion": 1,
    "by": 1,
    "came": 1,
    "can": 1,
    "cloud": 2,
    "converts": 1,
    "disk": 3,
    "dust": 1,
    "earth": 2,
    "earths": 1,
    "eventually": 1,
    "every": 1,
    "fit": 1,
    "flattened": 1,
    "form": 1,
    "formed": 2,
    "from": 1,
    "fuses": 1,
    "fusion": 1,
    "gas": 1,
    "gravity": 1,
    "has": 1,
    "helium": 2,
    "hundred": 1,
    "hydrogen": 2,
    "in": 2,
    "inside": 1,
    "into": 3,
    "is": 5,
    "it": 4,
    "kg": 1,
    "large": 1,
    "left": 1,
    "mainly": 1,
    "mass": 2,
    "material": 1,
    "million": 2,
    "molecular": 1,
    "moved": 1,
    "nuclear": 1,
    "of": 4,
    "old": 1,
    "orbiting": 1,
    "other": 1,
    "over": 1,
    "partly": 1,
    "planets": 3,
    "plus": 1,
    "second": 1,
    "solar": 1,
    "space": 1,
    "stars": 1,
    "sun": 6,
    "system": 1,
    "that": 2,
    "the": 14,
    "their": 1,
    "this": 3,
    "thought": 1,
    "through": 2,
    "times": 2,
    "together": 1,
    "tons": 1,
    "was": 1,
    "way": 1,
    "which": 1,
    "wide": 1,
    "years": 1,
}

In [ ]:
# Your solution here

In [ ]:
# Normalization and tokenization can be combined here because the target
# consists only of lower-case alphabetic tokens with at least two characters.
tokens = re.findall(r"[a-z]{2,}", text.lower())
vocabulary = dict(Counter(tokens))

print(vocabulary == target_vocabulary)


True


## **Ex. 2: Minimum edit distance**

Minimum edit distance measures how many editing operations are needed to transform a source string into a target string. Levenshtein distance assigns a cost of 1 to insertion, deletion, and substitution, and a cost of 0 when two characters are identical.

For example, the Levenshtein distance between `intention` and `execution` is 5.

A naive search over all edit sequences is expensive. Dynamic programming instead stores solutions to smaller prefix problems. If `D[i][j]` is the distance between `source[:i]` and `target[:j]`, each cell is computed from deletion, insertion, and substitution:

```text
D[i][j] = min(
    D[i-1][j]   + 1,
    D[i][j-1]   + 1,
    D[i-1][j-1] + substitution_cost,
)
```

See Section 2.9 of *Speech and Language Processing*.


In [ ]:
misspelled = ["hydrojen", "eearth", "moleculr", "gfgryg", "irf"]


**Question:** Implement Levenshtein distance using dynamic programming. Initialize the first row and column before filling the rest of the matrix.

<div class="alert alert-block alert-info">
<b>TIME:</b> About 20 minutes.
</div>


In [ ]:
def levenshtein(source: str, target: str) -> int:
    # Create and initialize the dynamic-programming matrix.
    ...

    # Fill the matrix using deletion, insertion, and substitution.
    ...

    # Return the distance between the complete strings.
    ...


In [ ]:
def levenshtein(source: str, target: str) -> int:
    rows = len(source) + 1
    columns = len(target) + 1
    distance = [[0] * columns for _ in range(rows)]

    for i in range(rows):
        distance[i][0] = i
    for j in range(columns):
        distance[0][j] = j

    for i in range(1, rows):
        for j in range(1, columns):
            substitution_cost = 0 if source[i - 1] == target[j - 1] else 1
            distance[i][j] = min(
                distance[i - 1][j] + 1,  # deletion
                distance[i][j - 1] + 1,  # insertion
                distance[i - 1][j - 1] + substitution_cost,
            )

    return distance[-1][-1]


In [ ]:
assert levenshtein("kitten", "sitting") == 3
assert levenshtein("intention", "execution") == 5

for typo in misspelled:
    options = [(candidate, levenshtein(typo, candidate)) for candidate in target_vocabulary]
    closest = min(options, key=lambda option: option[1])
    print(f"{typo} -> {closest}")


## **Ex. 3: Byte-Pair Encoding (BPE)**

Words are often useful units, but word vocabularies have difficulty with rare and unseen words.
One option is to use characters or bytes to avoid unknown words, but these produce much longer sequences. **Subword tokens** provide a compromise. A popular way of creating subwords is by using BPE; a compression algorithm.

BPE has two phases:

1. **Training:** begin with individual characters and repeatedly merge the most frequent adjacent pair.
2. **Encoding:** split new text into characters and apply the learned merges.

Following Section 2.4 of *Speech and Language Processing*, we first split the corpus into words and attach a visible whitespace marker (`▁`) to the beginning of each word. Pair counts are weighted by word frequency, and merges cannot cross word boundaries.

This implementation starts with Unicode characters for clarity. Production BPE tokenizers commonly start with UTF-8 bytes, which gives a fixed initial alphabet of 256 byte values and avoids unknown characters.

The implementation is adapted conceptually from [minbpe](https://github.com/karpathy/minbpe/blob/master/minbpe/basic.py), with word-level pretokenization added to match the chapter.

### Implement BPE

Complete the methods below:

- `merge`: replace every occurrence of a pair with a new token ID;
- `count_pairs`: count adjacent pairs, weighted by word frequency;
- `train`: repeatedly select and merge the most frequent pair; and
- `encode`: apply learned merges to new text in training order.

The corpus is represented as unique token sequences with frequencies. This is equivalent to counting over every occurrence, but avoids repeatedly scanning duplicate words.

<div class="alert alert-block alert-info">
<b>TIME:</b> About 30 minutes. The Hugging Face comparison below is optional.
</div>


In [ ]:
class BPETokenizer:
    def __init__(self, text: str, vocab_size: int):
        self.vocab_size = vocab_size
        self.marker = "▁"
        self.create_initial_vocab(text)

    def create_initial_vocab(self, text: str) -> None:
        words = [self.marker + word for word in text.split()]
        characters = sorted(set("".join(words)))

        self.text_to_id = {character: token_id for token_id, character in enumerate(characters)}
        self.id_to_text = {token_id: character for character, token_id in self.text_to_id.items()}
        self.words_to_merge = Counter(tuple(self.text_to_id[character] for character in word) for word in words)

    def merge(self, ids: tuple[int, ...], pair: tuple[int, int], new_id: int) -> tuple[int, ...]:
        # Replace every adjacent occurrence of pair with new_id.
        ...

    def count_pairs(self) -> Counter:
        # Count each adjacent pair, weighted by the word's frequency.
        ...

    def train(self) -> None:
        # Learn merges until the target vocabulary size is reached.
        ...

    def encode(self, text: str) -> list[int]:
        # Convert text to character IDs and apply learned merges in order.
        ...

    def decode(self, ids: list[int]) -> str:
        text = "".join(self.id_to_text[token_id] for token_id in ids)
        return text.replace(self.marker, " ").lstrip()


In [ ]:
class BPETokenizer:
    def __init__(self, text: str, vocab_size: int):
        self.vocab_size = vocab_size
        self.marker = "▁"
        self.create_initial_vocab(text)

    def create_initial_vocab(self, text: str) -> None:
        words = [self.marker + word for word in text.split()]
        characters = sorted(set("".join(words)))

        self.text_to_id = {character: token_id for token_id, character in enumerate(characters)}
        self.id_to_text = {token_id: character for character, token_id in self.text_to_id.items()}
        self.words_to_merge = Counter(tuple(self.text_to_id[character] for character in word) for word in words)

    def merge(self, ids: tuple[int, ...], pair: tuple[int, int], new_id: int) -> tuple[int, ...]:
        merged = []
        i = 0
        while i < len(ids):
            if i + 1 < len(ids) and (ids[i], ids[i + 1]) == pair:
                merged.append(new_id)
                i += 2
            else:
                merged.append(ids[i])
                i += 1
        return tuple(merged)

    def count_pairs(self) -> Counter:
        counts = Counter()
        print(self.words_to_merge)
        for ids, frequency in self.words_to_merge.items():
            for pair in zip(ids, ids[1:]):
                counts[pair] += frequency
        return counts

    def train(self) -> None:
        self.merges = {}
        number_of_merges = max(0, self.vocab_size - len(self.text_to_id))

        for _ in range(number_of_merges):
            counts = self.count_pairs()
            if not counts:
                break

            pair = max(counts, key=counts.get)
            new_id = len(self.id_to_text)
            token = self.id_to_text[pair[0]] + self.id_to_text[pair[1]]

            merged_words = Counter()
            for ids, frequency in self.words_to_merge.items():
                merged_words[self.merge(ids, pair, new_id)] += frequency
            self.words_to_merge = merged_words

            self.merges[pair] = new_id
            self.id_to_text[new_id] = token
            self.text_to_id[token] = new_id

    def encode(self, text: str) -> list[int]:
        characters = [character for word in text.split() for character in self.marker + word]
        ids = tuple(self.text_to_id[character] for character in characters)

        # Dictionaries retain insertion (merge) order
        for pair, new_id in self.merges.items():
            ids = self.merge(ids, pair, new_id)
        return list(ids)

    def decode(self, ids: list[int]) -> str:
        text = "".join(self.id_to_text[token_id] for token_id in ids)
        return text.replace(self.marker, " ").lstrip()


First test the algorithm on the small example from the chapter. It is useful to inspect the learned tokens before moving to a larger corpus.


In [ ]:
toy_corpus = "set new new renew reset renew"
toy_tokenizer = BPETokenizer(toy_corpus, vocab_size=13)
toy_tokenizer.train()

print(list(toy_tokenizer.text_to_id))
toy_ids = toy_tokenizer.encode("renew reset")
print([toy_tokenizer.id_to_text[token_id] for token_id in toy_ids])
print(toy_tokenizer.decode(toy_ids))


Counter({(6, 1, 0, 5): 2, (6, 2, 0, 1, 0, 5): 2, (6, 3, 0, 4): 1, (6, 2, 0, 3, 0, 4): 1})
Counter({(6, 7, 5): 2, (6, 2, 0, 7, 5): 2, (6, 3, 0, 4): 1, (6, 2, 0, 3, 0, 4): 1})
Counter({(6, 8): 2, (6, 2, 0, 8): 2, (6, 3, 0, 4): 1, (6, 2, 0, 3, 0, 4): 1})
Counter({(6, 8): 2, (9, 0, 8): 2, (6, 3, 0, 4): 1, (9, 0, 3, 0, 4): 1})
Counter({(6, 8): 2, (10, 8): 2, (6, 3, 0, 4): 1, (10, 3, 0, 4): 1})
Counter({(6, 8): 2, (10, 8): 2, (6, 11, 4): 1, (10, 11, 4): 1})
['e', 'n', 'r', 's', 't', 'w', '▁', 'ne', 'new', '▁r', '▁re', 'se', 'set']
['▁re', 'new', '▁re', 'set']
renew reset


Now train once on a larger Shakespeare corpus.


In [ ]:
text_corpus = " ".join(datasets.load_dataset("winglian/tiny-shakespeare")["train"]["text"])
print(f"Corpus size: {len(text_corpus):,} characters")
print(repr(text_corpus[:500]))

NameError: name 'datasets' is not defined

In [ ]:
tokenizer = BPETokenizer(text_corpus, vocab_size=200)
tokenizer.train()
print(f"Vocabulary size: {len(tokenizer.id_to_text)}")

In [ ]:
test_sentence = "Romeo, my love, have you heard about tokenization?"
token_ids = tokenizer.encode(test_sentence)
tokens = [tokenizer.id_to_text[token_id] for token_id in token_ids]

print(token_ids)
print(tokens)
print(tokenizer.decode(token_ids))


### Unknown characters

What happens when the character-level tokenizer encounters characters not seen during training? Why does starting from all 256 UTF-8 byte values "solve" this problem? What are some potential drawbacks?

See Section 2.3 from *Speech and Language Processing*.


In [ ]:
test_unknown = "你好，我不会说中文"
try:
    tokenizer.encode(test_unknown)
except KeyError as error:
    print(f"Unknown character: {error}")


The tokenizer can only encode characters in its initial vocabulary. A byte-level tokenizer first converts text to UTF-8 bytes. Since a byte has only 256 possible values, all input can be represented without an unknown-character token. BPE then learns frequent sequences of those bytes. The problem is that bytes are not necessarily "meaningful" units and different scripts are encoded unfairly: an ASCII character requires one byte (most Latin characters without accents), whereas a Chinese character requires three or even four.


## **Extension: Hugging Face Tokenizers**

The same training pipeline is available in the Hugging Face `tokenizers` library. The `Metaspace` replaces whitespace with a visible marker and prepends it to words, which is similar to our simplified pretokenization.

<div class="alert alert-block alert-info">
<b>TIME:</b> About 15 minutes. This section is optional.
</div>


In [ ]:
from tokenizers import Tokenizer, models, pre_tokenizers, trainers, decoders

hf_tokenizer = Tokenizer(models.BPE(unk_token="[UNK]"))
hf_tokenizer.pre_tokenizer = pre_tokenizers.Metaspace()
hf_tokenizer.decoder = decoders.Metaspace()
hf_trainer = trainers.BpeTrainer(vocab_size=200, special_tokens=["[UNK]"])

hf_tokenizer.train_from_iterator([text_corpus], trainer=hf_trainer)


In [ ]:
hf_tokenizer.pre_tokenizer.pre_tokenize_str("Let's test pre-tokenization!")


In [ ]:
encoding = hf_tokenizer.encode(test_sentence)
print(encoding.ids)
print(encoding.tokens)
print(hf_tokenizer.decode(encoding.ids))


Compare the custom and Hugging Face tokenizers:

- Do they produce the same tokens?
- How do they represent whitespace and punctuation?
- Which normalization, pretokenization, unknown-token, and decoding choices differ?
- Why can two BPE implementations with the same vocabulary size produce different vocabularies?

See the [Hugging Face tokenization pipeline documentation](https://huggingface.co/docs/tokenizers/en/pipeline).
